# nb12.5 — The Five-Year Research-Identity Map: A NetworkX Graph From One Paper to Three Futures

*Week 12, Chapter 12.5.* The Week-8 paper is not an endpoint. It is the first
node in a network that, with luck and discipline, grows into a research
identity -- a recognizable pattern of problems you ask, data you use, and
methods you bring. This notebook draws that network explicitly.

We build a `networkx` directed graph whose nodes are research projects (the
Week-8 paper plus three plausible five-year follow-ups), data sources, methods,
and target venues. Edges encode *uses* relations (a project uses a data source,
a project uses a method, a project targets a venue). We then run two analyses:
(a) shortest-path queries -- "what's the bridge between Maya's HMDA paper and
the Federal Reserve regulator network?" -- and (b) centrality -- "which datasets
and methods most define this researcher's identity?"


In [ ]:
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


## 1. The four projects

Maya's research identity, projected forward five years from her Week-8 paper:

- **`paper-0`** -- the Week-8 fair-lending paper, HMDA + Census, county fixed
  effects + IV using bank mergers.
- **`paper-1`** -- replicate the fair-lending pattern in auto lending, using
  CFPB consumer-complaints data plus state DMV registration files.
- **`paper-2`** -- extend to flood-zone lending, merging HMDA with FEMA flood
  maps to ask whether minority borrowers face premium gaps for the same
  expected flood exposure.
- **`paper-3`** -- text-analysis project on bank CRA (Community Reinvestment
  Act) public comment letters, using a fine-tuned BERT to score sentiment by
  comment-writer demographic.

The pattern is recognizable: she is a *fair-lending researcher*. Different
data, different methods, but the same identifying question.


In [ ]:
PROJECTS = [
    {"id": "paper-0", "name": "HMDA Fair-Lending (Week-8)",      "year": 2026,
     "stage": "submitted", "tag": "core"},
    {"id": "paper-1", "name": "Auto-Lending Disparities (Y+1)",  "year": 2027,
     "stage": "planned",   "tag": "replication-extension"},
    {"id": "paper-2", "name": "Flood-Zone Mortgages (Y+2)",      "year": 2028,
     "stage": "planned",   "tag": "data-extension"},
    {"id": "paper-3", "name": "CRA Comment-Letter Sentiment (Y+4)", "year": 2030,
     "stage": "planned",   "tag": "method-extension"},
]

DATA_SOURCES = [
    {"id": "data-hmda",     "name": "HMDA Loan Application Register"},
    {"id": "data-census",   "name": "US Census ACS"},
    {"id": "data-mergers",  "name": "FDIC Bank Merger Database"},
    {"id": "data-cfpb",     "name": "CFPB Consumer Complaints"},
    {"id": "data-dmv",      "name": "State DMV Registration Panels"},
    {"id": "data-fema",     "name": "FEMA National Flood Hazard Layer"},
    {"id": "data-cra-comments", "name": "Federal Reserve CRA Comment Letters"},
]

METHODS = [
    {"id": "method-fe",      "name": "Two-way fixed effects"},
    {"id": "method-iv",      "name": "Instrumental variables"},
    {"id": "method-did",     "name": "Difference-in-differences"},
    {"id": "method-rd",      "name": "Regression discontinuity (flood-zone boundaries)"},
    {"id": "method-bert",    "name": "Fine-tuned BERT classifier"},
]

VENUES = [
    {"id": "venue-jf",       "name": "Journal of Finance"},
    {"id": "venue-jfe",      "name": "Journal of Financial Economics"},
    {"id": "venue-jfqa",     "name": "Journal of Financial and Quantitative Analysis"},
    {"id": "venue-rfs",      "name": "Review of Financial Studies"},
    {"id": "venue-fdic",     "name": "FDIC Bank Research Conference"},
    {"id": "venue-fed-climate", "name": "Federal Reserve Climate-Risk Conference"},
    {"id": "venue-schar",    "name": "Schar Symposium"},
]

len(PROJECTS), len(DATA_SOURCES), len(METHODS), len(VENUES)


## 2. The edges

Edges encode three relations:

- `(project, data_source, uses_data)` -- what data does the project pull?
- `(project, method, uses_method)` -- what method does it apply?
- `(project, venue, targets)` -- where does the author plan to send it?

We hand-write the edges below; in a richer pipeline you would scrape them from
your own working-paper drafts using `regex` on data and method keywords.


In [ ]:
EDGES = [
    # paper-0: HMDA fair-lending
    ("paper-0", "data-hmda",    "uses_data"),
    ("paper-0", "data-census",  "uses_data"),
    ("paper-0", "data-mergers", "uses_data"),
    ("paper-0", "method-fe",    "uses_method"),
    ("paper-0", "method-iv",    "uses_method"),
    ("paper-0", "venue-schar",  "targets"),
    ("paper-0", "venue-fdic",   "targets"),

    # paper-1: auto-lending
    ("paper-1", "data-cfpb",    "uses_data"),
    ("paper-1", "data-dmv",     "uses_data"),
    ("paper-1", "data-census",  "uses_data"),
    ("paper-1", "method-fe",    "uses_method"),
    ("paper-1", "method-did",   "uses_method"),
    ("paper-1", "venue-jfqa",   "targets"),
    ("paper-1", "venue-fdic",   "targets"),

    # paper-2: flood-zone
    ("paper-2", "data-hmda",    "uses_data"),
    ("paper-2", "data-fema",    "uses_data"),
    ("paper-2", "data-census",  "uses_data"),
    ("paper-2", "method-fe",    "uses_method"),
    ("paper-2", "method-rd",    "uses_method"),
    ("paper-2", "venue-jfe",    "targets"),
    ("paper-2", "venue-fed-climate", "targets"),

    # paper-3: CRA letters
    ("paper-3", "data-cra-comments", "uses_data"),
    ("paper-3", "data-census",       "uses_data"),
    ("paper-3", "method-bert",       "uses_method"),
    ("paper-3", "venue-rfs",         "targets"),
    ("paper-3", "venue-jf",          "targets"),
]

len(EDGES)


## 3. Building the graph

We use `networkx.DiGraph` (directed graph) because the edges encode asymmetric
relations: a project *uses* a data source, not the other way around.


In [ ]:
G = nx.DiGraph()

# Add nodes with type attribute
for p in PROJECTS:
    G.add_node(p["id"], **p, node_type="project")
for d in DATA_SOURCES:
    G.add_node(d["id"], **d, node_type="data")
for m in METHODS:
    G.add_node(m["id"], **m, node_type="method")
for v in VENUES:
    G.add_node(v["id"], **v, node_type="venue")

# Add edges with relation attribute
for src, dst, rel in EDGES:
    G.add_edge(src, dst, relation=rel)

print(f"nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
node_counts = pd.Series(
    [G.nodes[n]["node_type"] for n in G.nodes]
).value_counts().rename("count").to_frame()
node_counts


## 4. Centrality: what defines this researcher's identity?

`networkx` provides several centrality measures. We use **in-degree centrality**
on the data and method nodes: a data source that *receives* edges from many
projects is central to the researcher's identity. The Census ACS, for example,
appears in three of four papers and is therefore the most central dataset in
Maya's research portfolio.


In [ ]:
centrality = pd.DataFrame([
    {"id": n, "name": G.nodes[n]["name"], "node_type": G.nodes[n]["node_type"],
     "in_degree": G.in_degree(n)}
    for n in G.nodes
    if G.nodes[n]["node_type"] in ("data", "method")
]).sort_values("in_degree", ascending=False)
centrality


**Reading the table.** The Census ACS and the two-way fixed effects estimator
are the central infrastructure of Maya's portfolio: every project except the
BERT paper uses both. That gives her a clean elevator pitch -- *"I am a
researcher who combines administrative loan-level microdata with panel
econometrics to ask whether the financial system treats demographically similar
borrowers similarly."* The portfolio defends that one-sentence identity.

**The lonely methods.** BERT and regression discontinuity each appear in
exactly one project. That is not a flaw -- it is the *direction of investment*.
Each represents a method Maya is buying into for the next stage of her career.


## 5. Shortest paths: bridges across the portfolio

Suppose you are at the FDIC Bank Research Conference and you want to know
which project in Maya's pipeline is the natural bridge to the Federal Reserve
Climate-Risk Conference. We ask `networkx` for the shortest path *from
`venue-fdic` to `venue-fed-climate`* in the undirected projection.


In [ ]:
UG = G.to_undirected()
try:
    path = nx.shortest_path(UG, source="venue-fdic", target="venue-fed-climate")
    path_names = [G.nodes[n]["name"] for n in path]
    for i, name in enumerate(path_names):
        prefix = "    " * i
        print(f"{prefix}- {name}")
except nx.NetworkXNoPath:
    print("no path")


The bridge runs through `paper-2` (the flood-zone project), which is exactly
the project Maya should mention if a Fed economist at the FDIC conference asks
her *"what are you working on next?"*. The graph encodes a piece of social
knowledge that would otherwise live only in her head.


## 6. Subgraph projection: just the projects-and-data network

A cleaner picture is to project away the venue nodes and look only at the
project-to-data relationships. This is the *empirical infrastructure* graph.


In [ ]:
data_project_nodes = [
    n for n in G.nodes
    if G.nodes[n]["node_type"] in ("project", "data")
]
H = G.subgraph(data_project_nodes).copy()
print(f"infrastructure subgraph: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")

# Which datasets are shared across the most projects?
data_to_projects = {}
for proj_id in (p["id"] for p in PROJECTS):
    for nbr in H.successors(proj_id):
        data_to_projects.setdefault(nbr, []).append(proj_id)

pd.DataFrame([
    {"data": G.nodes[d]["name"], "n_projects": len(ps), "projects": ", ".join(ps)}
    for d, ps in sorted(data_to_projects.items(), key=lambda x: -len(x[1]))
])


## 7. Drawing the graph

Visualizing the full graph with 23 nodes is messy. We instead render the
project-to-data subgraph with a deterministic layout, colored by node type.
The chart is for sanity-checking; the analyses above are the real outputs.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Use a deterministic layout: kamada-kawai is parameter-free and reproducible
pos = nx.kamada_kawai_layout(H)

# Color nodes by type
colors = ["#4C72B0" if G.nodes[n]["node_type"] == "project" else "#DD8452"
          for n in H.nodes]
nx.draw_networkx_nodes(H, pos, node_color=colors, node_size=500, ax=ax)
nx.draw_networkx_edges(H, pos, alpha=0.4, arrowsize=12, ax=ax)
labels = {n: G.nodes[n]["name"][:30] for n in H.nodes}
nx.draw_networkx_labels(H, pos, labels, font_size=7, ax=ax)
ax.set_title("Maya's project-to-data subgraph (five-year horizon)")
ax.axis("off")
plt.tight_layout()
fig.savefig("/tmp/nb12_5_project_data_graph.png", dpi=120)
plt.close(fig)
print("saved graph to /tmp/nb12_5_project_data_graph.png")


## 8. The identity statement

We end with a small text generator that summarizes the graph as a paragraph.
This is the thing Maya pastes into the "research statement" section of any
future application -- the part she will be writing five years from now.


In [ ]:
def identity_statement(G):
    proj_names = [G.nodes[n]["name"] for n in G.nodes if G.nodes[n]["node_type"] == "project"]
    # Most-central data and method
    central_data = sorted(
        [n for n in G.nodes if G.nodes[n]["node_type"] == "data"],
        key=lambda n: -G.in_degree(n),
    )[:2]
    central_methods = sorted(
        [n for n in G.nodes if G.nodes[n]["node_type"] == "method"],
        key=lambda n: -G.in_degree(n),
    )[:2]
    central_data_names = [G.nodes[n]["name"] for n in central_data]
    central_method_names = [G.nodes[n]["name"] for n in central_methods]
    return (
        f"My research portfolio (n={len(proj_names)} projects) sits at the intersection of "
        f"administrative microdata and panel econometrics. The empirical infrastructure I "
        f"return to most often is {' and '.join(central_data_names)}. Methodologically I "
        f"build on {' and '.join(central_method_names)}, extending into newer methods "
        f"(regression discontinuity, fine-tuned language models) as each project demands. "
        f"My current and planned projects are: " + "; ".join(proj_names) + "."
    )

print(identity_statement(G))


## 9. When it fails

**Premature commitment.** The five-year map is a *plan*, not a prediction. A
finding in paper-1 may make paper-2 uninteresting and pull a totally new
project into focus. Re-draw the graph annually.

**Single-method tunnel.** If every project in your portfolio uses fixed
effects and IV, the centrality table looks great but it is also evidence you
are not investing in new methods. The "lonely methods" (in-degree 1) should
not stay lonely; they should be the seeds of the next centrality table.

**Data-only portfolio.** A common failure for early-career researchers is to
let the portfolio be defined entirely by a single dataset (everyone's HMDA
papers). The fix is exactly what Maya is doing in paper-3: deliberately
introduce a dataset the rest of the portfolio does not touch.


## 10. Your turn

1. **Add a fifth project.** Hand-add a Y+5 project that closes the loop --
   perhaps a paper that uses HMDA, FEMA, and BERT together. Confirm the
   project-to-data subgraph becomes more interconnected.

2. **Devon's map.** Replicate the graph for Devon's stablecoin-remittance
   paper plus three plausible follow-ups. What are his central datasets?

3. **Co-author edges.** Add a fifth node type, `coauthor`, and edges
   `(project, coauthor, with)`. Compute betweenness centrality on the
   coauthor projection.

4. **Identity drift detector.** Write `identity_drift(G_t, G_t1)` that
   compares two snapshots of the portfolio graph and flags new central
   datasets or methods (in-degree rising by >= 2 between snapshots). This is
   the diagnostic you run on yourself every year of grad school.
